In [3]:
import numpy as np
import pandas as pd

In [6]:
ort = pd.read_csv("online_retail.csv")
ort.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [7]:
print(ort.isnull().sum())

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


In [9]:
# drop duplicates
ort.drop_duplicates(inplace=True)

In [13]:
# Invoice Number
ort["InvoiceNo"] = ort["InvoiceNo"].str.strip()
ort["IsCancellation"] = ort["InvoiceNo"].str.startswith("C")

In [14]:
# StockCode
ort["StockCode"] = ort["StockCode"].str.strip().str.upper()

In [15]:
# remove non-products
non_product_codes = {"POST", "DOT", "M", "AMAZONFEE", "BANK CHARGES", "PADS", "CRUK", "D", "C2"}
ort = ort[~ort["StockCode"].isin(non_product_codes)]

In [16]:
#description
ort["Description"] = ort["Description"].str.strip().str.title()

In [18]:
# Fill missing descriptions from the most common description per StockCode
desc_map = (
    ort.dropna(subset=["Description"])
    .groupby("StockCode")["Description"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
)
ort["Description"] = ort["Description"].fillna(ort["StockCode"].map(desc_map))

In [21]:
# Drop rows missing a description
ort.dropna(subset=["Description"], inplace=True)

In [20]:
# quantity
returns = ort[ort["Quantity"] <= 0].copy()
ort = ort[ort["Quantity"] > 0].copy()


In [19]:
# Cap extreme outliers (> 99.9th percentile),  flag not delete
q999 = ort["Quantity"].quantile(0.999)
ort["QuantityOutlier"] = ort["Quantity"] > q999

In [22]:
# invoicedate
ort["InvoiceDate"] = pd.to_datetime(ort["InvoiceDate"], errors="coerce")
ort.dropna(subset=["InvoiceDate"], inplace=True)
 
ort["Year"]  =ort["InvoiceDate"].dt.year
ort["Month"] = ort["InvoiceDate"].dt.month
ort["Day"]   =ort["InvoiceDate"].dt.day
ort["Hour"]  = ort["InvoiceDate"].dt.hour

In [23]:
# Unit price
ort["UnitPrice"] = pd.to_numeric(ort["UnitPrice"], errors="coerce")
ort = ort[ort["UnitPrice"] > 0]
 
p999 = ort["UnitPrice"].quantile(0.999)
ort["PriceOutlier"] = ort["UnitPrice"] > p999


In [24]:
# Tag guest transactions 
ort["IsGuest"] = ort["CustomerID"].isna()

In [25]:
# Country
ort["Country"] = ort["Country"].str.strip()
 
country_map = {
    "EIRE": "Ireland",
    "Eire": "Ireland",
    "RSA": "South Africa",
    "Channel Islands": "United Kingdom",
}
ort["Country"] = ort["Country"].replace(country_map)
ort = ort[ort["Country"] != "Unspecified"]


In [26]:
# Total Price
ort["TotalPrice"] = ort["Quantity"] * ort["UnitPrice"]

In [27]:
cols = [
    "InvoiceNo", "IsCancellation",
    "StockCode", "Description",
    "Quantity", "QuantityOutlier",
    "InvoiceDate", "Year", "Month", "Day", "Hour",
    "UnitPrice", "PriceOutlier",
    "TotalPrice",
    "CustomerID", "IsGuest",
    "Country",
]
ort = ort[cols]

In [28]:
output_path = "online_retail_cleaned.csv"
ort.to_csv(output_path, index=False)
returns.to_csv("online_retail_returns.csv", index=False)
 
print(f"Clean shape : {ort.shape}")
print(f"Returns rows: {len(returns)}")
print(f"Saved - {output_path}")
print(f"Saved - online_retail_returns.csv")

Clean shape : (522129, 17)
Returns rows: 10064
Saved - online_retail_cleaned.csv
Saved - online_retail_returns.csv
